In [ ]:
import subprocess
import sys

# Ép cài typing_extensions>=4.13 (bản có Sentinel) TRƯỚC, tách riêng khỏi lệnh cài
# các package khác — nếu gộp chung 1 lệnh, resolver có thể chọn bản typing_extensions
# thấp hơn do ràng buộc từ package khác. Dùng đúng sys.executable của kernel đang chạy
# để chắc chắn cài vào đúng môi trường (tránh lệch môi trường do %uv/%pip).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
     "typing_extensions>=4.13"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=4.51", "accelerate", "sentence-transformers",
     "qdrant-client>=1.10,<2", "pandas", "scikit-learn", "python-dotenv", "tqdm",
     "vllm==0.25.1"],
    check=True,
)


# torchcodec la dependency tuy chon (audio/video) bi keo theo qua transformers/vllm;
# eager-probe cua no hay crash vi thieu libnvrtc.so.13 dung CUDA runtime, khong lien
# quan gi toi pipeline text-only o day -> go het truoc khi restart kernel.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchcodec"],
    check=False,
)

import typing_extensions
from importlib.metadata import version as pkg_version
print('typing_extensions:', pkg_version('typing_extensions'), '| Sentinel import OK')
print('QUAN TRỌNG: giờ RESTART KERNEL rồi mới chạy các cell tiếp theo.')



In [1]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Tìm .env khi chạy tại root hoặc trong embedding/.
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path('.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.')

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
# ĐỔI MODEL: sửa đúng 1 dòng dưới đây (giữ nguyên đúng key trong MODEL_REPOS),
# rồi Restart Kernel + Run All. Không cần sửa gì khác để thử model kế tiếp.
AVAILABLE_MODELS = ['Qwen2.5-7B-Instruct']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v2'
BENCHMARK_PROTOCOL = 'vendor_recommended_multi_seed'
BGE_MODEL_ID = 'BAAI/bge-m3'
QDRANT_COLLECTION = 'laws_bge_m3_v2_correct_pooling'
EXPECTED_VECTOR_DIM = 1024
TOP_K = 10
MAX_INPUT_TOKENS = 24000
MAX_NEW_TOKENS = 8192
MAX_ARTICLE_CHARS = 6000  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # benchmark strict: không retry để chọn output hợp lệ hơn
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    # Modal Notebook: secret được attach lúc tạo notebook -> đã có sẵn trong os.environ,
    # không cần bước nào khác. Fallback Kaggle Secrets chỉ kích hoạt khi chạy trên Kaggle.
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
                if value:
                    return value
            except Exception:
                pass
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Thiếu secret, cần một trong: {names}')
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)
# Khong con dung Qdrant (retrieval nap tu file retrieval_top14_ours.json) -> khong bat buoc nua.
QDRANT_URL = get_secret('QDRANT_URL', required=False)
QDRANT_API_KEY = get_secret('QDRANT_API_KEY', 'QDRANT_KEY', required=False)

assert torch.cuda.is_available(), 'Notebook này yêu cầu GPU CUDA — khi tạo Modal Notebook nhớ chọn GPU (A10G trở lên cho model 7-8B).'
torch.manual_seed(EVAL_SEEDS[0])
torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('GPU:', torch.cuda.get_device_name(0))
print('Model:', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('Precision:', 'unquantized', 'BF16' if torch.cuda.is_bf16_supported() else 'FP16')


Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.
GPU: NVIDIA L4
Model: Qwen2.5-7B-Instruct -> Qwen/Qwen2.5-7B-Instruct
Benchmark: v2 | protocol: vendor_recommended_multi_seed
Generation profile: vendor_qwen2_5_instruct | seeds: [2026]
Precision: unquantized BF16
Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.
GPU: NVIDIA L4
Model: Qwen2.5-7B-Instruct -> Qwen/Qwen2.5-7B-Instruct
Benchmark: v2 | protocol: vendor_recommended_multi_seed
Generation profile: vendor_qwen2_5_instruct | seeds: [2026]
Precision: unquantized BF16


In [3]:
def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PUBLIC_TEST_PATH=/path/to/file.json
    candidates = []
    if os.getenv('ALQAC_PUBLIC_TEST_PATH'):
        candidates.append(Path(os.environ['ALQAC_PUBLIC_TEST_PATH']))
    candidates += [
        # Modal Notebook: upload ALQAC2026_public_test.json qua file browser bên trái
        # (kéo thả vào đúng thư mục làm việc của notebook) -> sẽ khớp 1 trong 2 dòng dưới.
        Path('ALQAC2026_public_test.json'),
        Path('data/ALQAC2026_public_test.json'),
        Path('../data/ALQAC2026_public_test.json'),
        # Kaggle (giữ lại để notebook vẫn chạy được trên Kaggle nếu cần đối chiếu).
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh/ALQAC2026_public_test.json'),
        Path('/kaggle/working/ALQAC2026_public_test.json'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('ALQAC2026_public_test.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(
        f'Không tìm thấy ALQAC2026_public_test.json. Đã kiểm tra:\n{checked}\n'
        f'Trên Modal Notebook: upload file này qua file browser, hoặc set '
        f"os.environ['ALQAC_PUBLIC_TEST_PATH'] ở cell trước khi gọi find_public_test()."
    )

DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

assert len(public_data) == 50, f'Expected 50 cases, got {len(public_data)}'
assert len({x['case_id'] for x in public_data}) == len(public_data)
assert all(x.get('case_query') for x in public_data)
assert all(x.get('verdict_label') in LABELS for x in public_data)

# ---- Nạp CHUNK (agent_v4_results.json) làm case_facts bổ sung ngoài case_query ----
def find_data_file(name):
    candidates = []
    if os.getenv('ALQAC_' + name.upper().replace('.', '_') + '_PATH'):
        candidates.append(Path(os.environ['ALQAC_' + name.upper().replace('.', '_') + '_PATH']))
    candidates += [
        Path(name), Path('outputs') / name, Path('data') / name,
        Path('../outputs') / name, Path('../data') / name,
        Path('/kaggle/working') / name,
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(name))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f'Không tìm thấy {name}.')

AGENT_EVIDENCE_PATH = find_data_file('agent_v4_results.json')
with AGENT_EVIDENCE_PATH.open(encoding='utf-8') as f:
    agent_evidence_raw = json.load(f)

MAX_EVIDENCE_CHARS = int(os.getenv('ALQAC_MAX_EVIDENCE_CHARS', '8000'))

def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
    seen, parts, total = set(), [], 0
    for e in record.get('evidence_details', []):
        chunk_id = e.get('chunk_id')
        txt = str(e.get('text', '')).strip()
        if not txt or chunk_id in seen:
            continue
        seen.add(chunk_id)
        if total + len(txt) + 1 > max_chars:
            break
        parts.append(txt)
        total += len(txt) + 1
    return '\n'.join(parts)

evidence_by_case = {r['case_id']: _case_evidence_text(r) for r in agent_evidence_raw}
_missing_ev = [x['case_id'] for x in public_data if not evidence_by_case.get(x['case_id'])]
if _missing_ev:
    print('CẢNH BÁO: thiếu evidence cho case:', _missing_ev)
print('Evidence bổ sung:', AGENT_EVIDENCE_PATH, '| phủ', len(public_data) - len(_missing_ev), '/', len(public_data), 'case')

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
inference_cases = [
    {'case_id': x['case_id'], 'case_query': x['case_query'],
     'case_facts': evidence_by_case.get(x['case_id'], '')}
    for x in public_data
]
gold_by_case = {x['case_id']: x['verdict_label'] for x in public_data}

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dataset: /root/ALQAC2026_public_test.json
Cases: 50
Qdrant: laws_bge_m3_v2_correct_pooling | dim: 1024 | points: 3352


In [ ]:
# ---- Nạp SẴN điều luật/case từ retrieval_top14_ours.json (KHÔNG embed/search lại) ----
# File này = legal_query rewrite + hybrid dense+BM25 + inject điều thủ tục, đã tính
# trước cho cả 50 vụ public (xem alqac2026/make_retrieval_top10.py). Thay hẳn cho
# retrieve_laws() cũ (BGE-M3 + Qdrant live) -> khỏi cần Qdrant/embedder nữa, nhanh hơn nhiều.
RETRIEVAL_PATH = find_data_file('retrieval_top14_ours.json')
with RETRIEVAL_PATH.open(encoding='utf-8') as f:
    _retrieval_raw = json.load(f)

retrieval_cache = {}
for case_id, laws in _retrieval_raw.items():
    retrieval_cache[case_id] = [
        {
            'rank': int(item['rank']), 'score': float(item.get('score', 0.0)),
            'law_id': str(item['law_id']), 'aid': int(item['aid']),
            'article_no': int(item['article_no']),
            'content_Article': str(item.get('content_Article') or ''),
        }
        for item in laws
    ]

expected_case_ids = {x['case_id'] for x in inference_cases}
missing = expected_case_ids - set(retrieval_cache)
assert not missing, f'Thiếu retrieval cho case: {sorted(missing)}'
law_counts = {len(v) for v in retrieval_cache.values()}
assert len(law_counts) == 1, f'Số điều/vụ không đồng nhất: {law_counts}'
TOP_K = next(iter(law_counts))  # cập nhật TOP_K theo đúng số điều thật trong file (14, không phải 10)

# Fingerprint cua file retrieval tinh (thay cho signature cua embedding song ban goc)
# -- dung trong BENCHMARK_MANIFEST/cache_key de nhan dien dung bo du lieu retrieval nao.
retrieval_signature = hashlib.sha256(RETRIEVAL_PATH.read_bytes()).hexdigest()

sample_case = inference_cases[0]
sample_laws = retrieval_cache[sample_case['case_id']]
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])
print('Retrieval:', RETRIEVAL_PATH, '| case:', len(retrieval_cache), '| điều/vụ:', TOP_K,
      '| signature:', retrieval_signature[:12])

In [ ]:
# Load model qua vLLM (batch nhiều case cùng lúc -> nhanh hơn transformers.generate() tuần tự).
MODEL_DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# flashinfer JIT-compile kernel sampler co the FAIL neu moi truong thieu CUDA dev
# headers (vd. curand.h khong ton tai) -> ep dung sampler PyTorch thuan (khong can
# JIT compile) va tat CUDA graph capture/torch.compile (enforce_eager) de tranh moi
# buoc "compile_or_warm_up_model" co the crash o day. Doi lai chut toc do, doi lay
# on dinh (khong crash EngineCore).
os.environ.setdefault('VLLM_USE_FLASHINFER_SAMPLER', '0')

from vllm import LLM, SamplingParams

vllm_engine = LLM(
    model=MODEL_ID,
    dtype=MODEL_DTYPE,
    trust_remote_code=True,
    max_model_len=MAX_INPUT_TOKENS + MAX_NEW_TOKENS,
    max_num_seqs=64,
    gpu_memory_utilization=0.92,
    enforce_eager=True,
    seed=EVAL_SEEDS[0],
)
MODEL_REVISION = 'vllm-' + MODEL_ID
RESOLVED_GENERATION_CONFIG = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items()}
print('Loaded via vLLM:', MODEL_ID, '| dtype:', MODEL_DTYPE)
print('Resolved generation config:', json.dumps(RESOLVED_GENERATION_CONFIG, ensure_ascii=False))

SYSTEM_PROMPT = '''Bạn là chuyên gia phân tích tranh chấp dân sự Việt Nam.
Bạn chỉ được sử dụng CASE_QUERY và 10 ĐIỀU LUẬT được cung cấp. Không được giả định dữ kiện ngoài đầu vào.
A là nguyên đơn, B là bị đơn. Hãy dự đoán đúng một trong bốn nhãn:
- A_WIN: toàn bộ hoặc về cơ bản toàn bộ yêu cầu của nguyên đơn được chấp nhận.
- B_WIN: yêu cầu của nguyên đơn bị bác toàn bộ hoặc về cơ bản toàn bộ.
- PARTIAL_A_WIN: nguyên đơn được chấp nhận một phần đáng kể nhưng không toàn bộ; kết quả nghiêng về A.
- PARTIAL_B_WIN: có phần yêu cầu của nguyên đơn được chấp nhận nhưng kết quả chủ yếu nghiêng về B.

Trả về đúng một JSON object, không Markdown, không văn bản bên ngoài JSON:
{
  "prediction": "<LABEL>",
  "confidence": 0.78,
  "reasoning": "Lập luận ngắn gọn bằng tiếng Việt",
  "applied_laws": [
    {"law_id": "91/2015/QH13", "aid": 53373, "reason": "Lý do áp dụng"}
  ]
}
Thay <LABEL> bằng đúng một trong A_WIN, B_WIN, PARTIAL_A_WIN, PARTIAL_B_WIN; không được giữ placeholder.
Chỉ chọn applied_laws từ danh sách 10 điều luật. Confidence phải nằm trong [0, 1].'''

BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'model_name': MODEL_NAME, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'generation_profile': ACTIVE_PROFILE,
    'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
    'dtype': str(MODEL_DTYPE), 'full_gpu_no_quantization': True,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_total_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': package_version('transformers'),
    'sentence_transformers': package_version('sentence-transformers'),
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases), 'labels': LABELS,
    'bge_model': BGE_MODEL_ID, 'qdrant_collection': QDRANT_COLLECTION, 'top_k_laws': TOP_K,
    'max_input_tokens': MAX_INPUT_TOKENS, 'max_new_tokens': MAX_NEW_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    'eval_seeds': EVAL_SEEDS,
    'invalid_output_policy': 'count_as_wrong',
    'system_prompt_sha256': hashlib.sha256(SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    'retrieval_signature': retrieval_signature,
}

def build_user_prompt(case_query, laws, case_facts=''):
    law_blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        law_blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f"{content}"
        )
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return (
        'CASE_QUERY:\n' + case_query.strip() + facts_block +
        f'\n\n{len(laws)} ĐIỀU LUẬT TRUY XUẤT:\n' + '\n\n'.join(law_blocks) +
        '\n\nHãy phân tích và trả về đúng JSON schema đã yêu cầu.'
    )

def build_messages(user_prompt):
    if ACTIVE_PROFILE['use_system_prompt']:
        return [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ]
    # DeepSeek-R1 khuyến nghị không dùng system role; nội dung hướng dẫn vẫn giữ nguyên.
    return [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user_prompt}]

def extract_first_json(text):
    text = re.sub(r'<think>.*?</think>', '', text or '', flags=re.I | re.S).strip()
    if '</think>' in text:
        text = text.rsplit('</think>', 1)[-1].strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.I | re.S).strip()
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    try:
        obj = ast.literal_eval(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    raise ValueError('Không tìm thấy JSON object hợp lệ')

def validate_prediction(obj, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'prediction không hợp lệ: {prediction!r}')
    confidence = float(obj.get('confidence'))
    if not 0.0 <= confidence <= 1.0:
        raise ValueError(f'confidence ngoài [0,1]: {confidence}')
    reasoning = str(obj.get('reasoning', '')).strip()
    if not reasoning:
        raise ValueError('reasoning rỗng')
    allowed = {(x['law_id'], int(x['aid'])) for x in retrieved_laws}
    clean_laws = []
    seen = set()
    for item in obj.get('applied_laws', []):
        try:
            key = (str(item['law_id']), int(item['aid']))
        except Exception:
            continue
        if key not in allowed or key in seen:
            continue
        seen.add(key)
        clean_laws.append({
            'law_id': key[0], 'aid': key[1],
            'reason': str(item.get('reason', '')).strip(),
        })
    return {
        'prediction': prediction,
        'confidence': confidence,
        'reasoning': reasoning,
        'applied_laws': clean_laws,
    }

class PredictionFormatError(RuntimeError):
    def __init__(self, message, raw_response, usage):
        super().__init__(message)
        self.raw_response = raw_response
        self.usage = usage

def _render_prompt(case_query, retrieved_laws, case_facts=''):
    user_prompt = build_user_prompt(case_query, retrieved_laws, case_facts)
    messages = build_messages(user_prompt)
    template_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        template_kwargs['enable_thinking'] = ACTIVE_PROFILE['enable_thinking']
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, **template_kwargs
    )
    input_tokens = len(tokenizer(rendered, truncation=False)['input_ids'])
    if input_tokens > MAX_INPUT_TOKENS:
        raise ValueError(f'Input vượt budget: {input_tokens}/{MAX_INPUT_TOKENS} tokens')
    return rendered, user_prompt, input_tokens

def _sampling_params_for(case_query, eval_seed):
    case_seed = eval_seed + int(hashlib.sha256(case_query.encode('utf-8')).hexdigest()[:8], 16)
    kwargs = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items() if k != 'do_sample'}
    return SamplingParams(max_tokens=MAX_NEW_TOKENS, seed=case_seed % (2**31 - 1), **kwargs), case_seed

def call_local_llm_batch(model_name, batch):
    """BATCH nhiều case trong 1 lệnh generate của vLLM (thay vòng lặp tuần tự cũ).
    batch = [(case_query, retrieved_laws, case_facts, eval_seed), ...]
    Trả list cùng thứ tự: (parsed_hoặc_None, raw_text, usage, user_prompt, error_hoặc_None)."""
    assert model_name == MODEL_NAME
    prompts, sampling_list, meta = [], [], []
    results = [None] * len(batch)
    for idx, (case_query, retrieved_laws, case_facts, eval_seed) in enumerate(batch):
        assert eval_seed in EVAL_SEEDS
        try:
            rendered, user_prompt, input_tokens = _render_prompt(case_query, retrieved_laws, case_facts)
        except Exception as exc:
            results[idx] = (None, None, None, None, exc)
            continue
        sp, case_seed = _sampling_params_for(case_query, eval_seed)
        prompts.append(rendered)
        sampling_list.append(sp)
        meta.append((idx, input_tokens, case_seed, user_prompt))

    if prompts:
        started = time.time()
        outputs = vllm_engine.generate(prompts, sampling_list)
        batch_duration = time.time() - started
        for (idx, input_tokens, case_seed, user_prompt), out in zip(meta, outputs):
            case_query, retrieved_laws, case_facts, eval_seed = batch[idx]
            gen = out.outputs[0]
            raw_text = (gen.text or '').strip()
            output_tokens = len(gen.token_ids)
            usage = {
                'input_tokens': input_tokens, 'output_tokens': output_tokens,
                'total_tokens': input_tokens + output_tokens,
                'hit_max_new_tokens': gen.finish_reason == 'length',
                'eval_seed': eval_seed, 'case_seed': case_seed,
                'batch_duration_seconds': round(batch_duration, 3),
            }
            if not raw_text:
                results[idx] = (None, raw_text, usage, user_prompt,
                                PredictionFormatError('Model trả output rỗng', raw_text, usage))
                continue
            try:
                parsed = validate_prediction(extract_first_json(raw_text), retrieved_laws)
                results[idx] = (parsed, raw_text, usage, user_prompt, None)
            except Exception as exc:
                results[idx] = (None, raw_text, usage, user_prompt,
                                PredictionFormatError(str(exc), raw_text, usage))
    return results


In [6]:
smoke_model = MODELS_TO_RUN[0]
smoke_seed = EVAL_SEEDS[0]
smoke_laws = retrieval_cache[sample_case['case_id']]
smoke_facts = sample_case.get('case_facts', '')
try:
    [(smoke_result, smoke_raw, smoke_usage, _smoke_prompt, smoke_err)] = call_local_llm_batch(
        smoke_model, [(sample_case['case_query'], smoke_laws, smoke_facts, smoke_seed)]
    )
    if smoke_err is not None:
        raise smoke_err
    print('Model:', smoke_model, '| seed:', smoke_seed)
    print(json.dumps(smoke_result, ensure_ascii=False, indent=2))
    print('Usage:', smoke_usage)
except Exception as exc:
    # Smoke lỗi không làm dừng benchmark; batch vẫn chấm case này đúng một lần theo seed.
    print('SMOKE WARNING:', repr(exc))

Model: Qwen2.5-7B-Instruct | seed: 2026
{
  "prediction": "A_WIN",
  "confidence": 0.85,
  "reasoning": "The case involves a personal injury and property damage caused by a dog running loose. According to Article 603 of the Civil Code (Law 91/2015/QH13), the owner of the animal must compensate for damages caused by the animal. Since the dog was running loose and caused the accident, the owners (B) are liable for compensation. Therefore, it is reasonable to predict that the plaintiff (A) will win the case as her claims for medical expenses and vehicle repair costs are likely to be supported.",
  "applied_laws": [
    {
      "law_id": "91/2015/QH13",
      "aid": 53373,
      "reason": "Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác. Người chiếm hữu, sử dụng súc vật phải bồi thường thiệt hại trong thời gian chiếm hữu, sử dụng súc vật, trừ trường hợp có thỏa thuận khác."
    }
  ]
}
Usage: {'input_tokens': 1666, 'output_tokens': 239, 'total_tokens': 1905, '

In [7]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')

def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default

def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'generation_profile': ACTIVE_PROFILE,
        'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'], 'case_facts': case.get('case_facts', ''), 'laws': laws,
        'system_prompt': SYSTEM_PROMPT,
        'max_input_tokens': MAX_INPUT_TOKENS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    }
    raw = json.dumps(material, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

assert len(retrieval_cache) == len(inference_cases)

# all_model_results[model][seed][case_id] -> result
all_model_results = {}
for model in MODELS_TO_RUN:
    all_model_results[model] = {}
    for eval_seed in EVAL_SEEDS:
        result_path = OUTPUT_DIR / f'predictions_{slugify(model)}_seed-{eval_seed}.json'
        saved = load_json(result_path, {})
        print(f'\n=== {model} | seed={eval_seed} | cached {len(saved)}/{len(inference_cases)} ===')

        pending = []  # [(case, laws, cache_key), ...] - case CHUA co cache hop le
        for case in inference_cases:
            case_id = case['case_id']
            laws = retrieval_cache[case_id]
            cache_key = make_cache_key(model, case, laws, eval_seed)
            old = saved.get(case_id)
            # Cache cả output lỗi: không cho case thêm cơ hội chỉ vì lần trước sai format.
            if old and old.get('cache_key') == cache_key:
                continue
            pending.append((case, laws, cache_key))

        print(f'  Cần tính: {len(pending)}/{len(inference_cases)} case '
              f'(1 lệnh generate() batch qua vLLM thay vì {len(pending)} lệnh tuần tự)')

        if pending:
            batch_input = [
                (case['case_query'], laws, case.get('case_facts', ''), eval_seed)
                for case, laws, _cache_key in pending
            ]
            started = time.time()
            batch_results = call_local_llm_batch(model, batch_input)
            batch_duration = round(time.time() - started, 3)
            per_case_duration = round(batch_duration / max(1, len(pending)), 3)

            for (case, laws, cache_key), (parsed, raw_text, usage, _prompt, err) in zip(pending, batch_results):
                case_id = case['case_id']
                if err is None:
                    saved[case_id] = {
                        'case_id': case_id,
                        'case_query': case['case_query'],
                        'eval_seed': eval_seed,
                        **parsed,
                        'retrieved_laws': laws,
                        'raw_response': raw_text,
                        'usage': usage,
                        'duration_seconds': per_case_duration,
                        'generation_attempts': 1,
                        'cache_key': cache_key,
                        'error': None,
                    }
                else:
                    saved[case_id] = {
                        'case_id': case_id,
                        'case_query': case['case_query'],
                        'eval_seed': eval_seed,
                        'prediction': None,
                        'confidence': None,
                        'reasoning': '',
                        'applied_laws': [],
                        'retrieved_laws': laws,
                        'raw_response': getattr(err, 'raw_response', raw_text),
                        'usage': getattr(err, 'usage', usage),
                        'duration_seconds': per_case_duration,
                        'generation_attempts': 1,
                        'cache_key': cache_key,
                        'error': repr(err),
                    }
                    print(f'  INVALID {case_id} | seed={eval_seed}: {err}')
            atomic_write_json(result_path, saved)
            print(f'  Batch xong trong {batch_duration}s (~{per_case_duration}s/case).')
        all_model_results[model][eval_seed] = saved

print('Hoàn tất batch cho', len(EVAL_SEEDS), 'seed x', len(inference_cases), 'case.')


Benchmark manifest: outputs_alqac_e2e/v2/benchmark_manifest_qwen2-5-7b-instruct.json

=== Qwen2.5-7B-Instruct | seed=2026 | cached 0/50 ===


Qwen2.5-7B-Instruct seed=2026:   0%|          | 0/50 [00:00<?, ?it/s]

Hoàn tất batch cho 1 seed x 50 case.


In [8]:
def evaluate_run(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        prediction = item.get('prediction')
        is_valid_output = prediction in LABELS
        usage = item.get('usage') or {}
        rows.append({
            'model': model,
            'seed': eval_seed,
            'case_id': cid,
            'gold': gold_by_case[cid],
            'prediction': prediction,
            'scored_prediction': prediction if is_valid_output else INVALID_OUTPUT_LABEL,
            'is_valid_output': is_valid_output,
            'confidence': item.get('confidence'),
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'duration_seconds': item.get('duration_seconds'),
            'error': item.get('error'),
        })
    frame = pd.DataFrame(rows)
    valid = frame[frame['is_valid_output']].copy()
    n_total, n_valid = len(frame), len(valid)
    n_failed = n_total - n_valid
    if n_failed:
        failed_ids = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
        print(
            f'Cảnh báo {model} seed={eval_seed}: {n_failed}/{n_total} output không hợp lệ '
            f'được tính sai. Case: {failed_ids}'
        )

    scored_prediction = frame['scored_prediction']
    strict_correct = int((frame['gold'] == scored_prediction).sum())
    strict_accuracy = strict_correct / n_total if n_total else 0.0
    valid_accuracy = accuracy_score(valid['gold'], valid['prediction']) if n_valid else 0.0
    report = classification_report(
        frame['gold'], scored_prediction, labels=LABELS,
        output_dict=True, zero_division=0,
    ) if n_total else {}
    cm_all = confusion_matrix(
        frame['gold'], scored_prediction, labels=LABELS + [INVALID_OUTPUT_LABEL]
    ) if n_total else np.zeros((len(LABELS) + 1, len(LABELS) + 1), dtype=int)
    cm = cm_all[:len(LABELS), :]
    summary = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model,
        'seed': eval_seed,
        'n_total': n_total,
        'n_success': n_valid,
        'n_failed': n_failed,
        'invalid_output_rate': n_failed / n_total if n_total else 0.0,
        'coverage': n_valid / n_total if n_total else 0.0,
        'benchmark_valid': n_total == len(inference_cases),
        'all_outputs_valid': n_valid == n_total,
        'metric_scope': 'all_50_invalid_outputs_count_as_wrong',
        'strict_accuracy_all_50': strict_accuracy,
        'accuracy_successful_only': valid_accuracy,
        'macro_precision': report.get('macro avg', {}).get('precision', 0.0),
        'macro_recall': report.get('macro avg', {}).get('recall', 0.0),
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0.0),
        'weighted_f1': report.get('weighted avg', {}).get('f1-score', 0.0),
        'avg_input_tokens': frame['input_tokens'].mean(),
        'avg_output_tokens': frame['output_tokens'].mean(),
        'avg_duration_seconds': frame['duration_seconds'].mean(),
        'n_hit_max_new_tokens': int(frame['hit_max_new_tokens'].sum()),
    }
    per_label = pd.DataFrame([
        {
            'model': model,
            'seed': eval_seed,
            'label': label,
            'precision': report.get(label, {}).get('precision', 0.0),
            'recall': report.get(label, {}).get('recall', 0.0),
            'f1': report.get(label, {}).get('f1-score', 0.0),
            'support': int(report.get(label, {}).get('support', 0)),
        } for label in LABELS
    ])
    cm_frame = pd.DataFrame(
        cm,
        index=[f'gold_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS + [INVALID_OUTPUT_LABEL]],
    )
    return summary, per_label, cm_frame, frame

summaries = []
evaluation_artifacts = {}
for model, seed_results in all_model_results.items():
    evaluation_artifacts[model] = {}
    for eval_seed, results in seed_results.items():
        summary, per_label, cm_frame, case_frame = evaluate_run(model, eval_seed, results)
        summaries.append(summary)
        evaluation_artifacts[model][eval_seed] = {
            'per_label': per_label,
            'confusion_matrix': cm_frame,
            'cases': case_frame,
        }
        print(f'\n=== {model} | seed={eval_seed} ===')
        display(pd.DataFrame([summary]))
        display(cm_frame)

run_metrics = pd.DataFrame(summaries).sort_values(['model', 'seed']).reset_index(drop=True)
aggregate_metrics = run_metrics.groupby('model', as_index=False).agg(
    n_seeds=('seed', 'nunique'),
    accuracy_mean=('strict_accuracy_all_50', 'mean'),
    accuracy_std=('strict_accuracy_all_50', 'std'),
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    invalid_rate_mean=('invalid_output_rate', 'mean'),
    invalid_rate_std=('invalid_output_rate', 'std'),
    avg_output_tokens=('avg_output_tokens', 'mean'),
    avg_duration_seconds=('avg_duration_seconds', 'mean'),
)
aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']] = (
    aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']].fillna(0.0)
)
leaderboard = aggregate_metrics.sort_values(
    ['accuracy_mean', 'macro_f1_mean'], ascending=False
).reset_index(drop=True)

majority_label = pd.Series(list(gold_by_case.values())).value_counts().idxmax()
majority_accuracy = pd.Series(list(gold_by_case.values())).value_counts().max() / len(gold_by_case)
print(f'Majority baseline: {majority_label} | accuracy={majority_accuracy:.4f}')
print('Per-seed metrics:')
display(run_metrics)
print('Aggregate mean ± std across seeds:')
display(leaderboard)

for model, seed_artifacts in evaluation_artifacts.items():
    slug = slugify(model)
    model_runs = run_metrics[run_metrics['model'] == model]
    model_summary = leaderboard[leaderboard['model'] == model]
    model_runs.to_csv(
        OUTPUT_DIR / f'model_metrics_by_seed_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    model_summary.to_csv(
        OUTPUT_DIR / f'model_metrics_summary_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    for eval_seed, artifacts in seed_artifacts.items():
        suffix = f'{slug}_seed-{eval_seed}'
        artifacts['per_label'].to_csv(
            OUTPUT_DIR / f'metrics_per_label_{suffix}.csv', index=False, encoding='utf-8-sig'
        )
        artifacts['confusion_matrix'].to_csv(
            OUTPUT_DIR / f'confusion_matrix_{suffix}.csv', encoding='utf-8-sig'
        )
        artifacts['cases'].to_csv(
            OUTPUT_DIR / f'case_predictions_{suffix}.csv', index=False, encoding='utf-8-sig'
        )



=== Qwen2.5-7B-Instruct | seed=2026 ===


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,Qwen2.5-7B-Instruct,2026,50,50,0,0.0,1.0,True,...,0.28,0.28,0.134868,0.191612,0.150063,0.215238,3267.3,276.4,18.02278,0


,pred_A_WIN,pred_B_WIN,pred_PARTIAL_A_WIN,pred_PARTIAL_B_WIN,pred___INVALID_OUTPUT__
gold_A_WIN,3,0,13,0,0
gold_B_WIN,0,0,10,0,0
gold_PARTIAL_A_WIN,8,0,11,0,0
gold_PARTIAL_B_WIN,1,0,4,0,0


Majority baseline: PARTIAL_A_WIN | accuracy=0.3800
Per-seed metrics:


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,Qwen2.5-7B-Instruct,2026,50,50,0,0.0,1.0,True,...,0.28,0.28,0.134868,0.191612,0.150063,0.215238,3267.3,276.4,18.02278,0


Aggregate mean ± std across seeds:


,model,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,invalid_rate_std,avg_output_tokens,avg_duration_seconds
0,Qwen2.5-7B-Instruct,1,0.28,0.0,0.150063,0.0,0.0,0.0,276.4,18.02278


In [9]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                'case_evidence': [],
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in item.get('applied_laws', [])
                ],
            })
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| evaluated:', len(inference_cases), '/ 50',
            '| invalid counted wrong:', n_failed,
            '| valid submission rows:', len(submission), '/ 50',
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


Qwen2.5-7B-Instruct | seed: 2026 | evaluated: 50 / 50 | invalid counted wrong: 0 | valid submission rows: 50 / 50 | outputs_alqac_e2e/v2/submission_qwen2-5-7b-instruct_seed-2026.json
Outputs: /root/outputs_alqac_e2e/v2
